# Search or Think? Reproduce the evaluation results

**Runtime:** GPU, **NVIDIA L4** (Runtime, Change runtime type, L4 GPU). Every result in the report was produced on an L4 with vLLM 0.31.0 and TRL 1.16.0, and this notebook installs the same versions. The policies run in bfloat16, so the free T4 GPU cannot run them. No Google Drive is needed, and no training is run.

**What it does.** Runs `reproduce.py` from the repository:

1. checks the GPU and the library versions;
2. runs the unit tests (`test_sot.py`, `test_export_site.py`, `test_reproduce.py`);
3. starts the BM25 search server over Pyserini's prebuilt `wikipedia-dpr-100w` index (a download of several GB), then searches a sample of 2,000 queries from the recorded runs again and checks that the same top 3 passages come back;
4. downloads the untrained Qwen2.5-0.5B-Instruct from the Hugging Face Hub and the five trained policies from this repository's release: the start policy (stage A2), plain GRPO, fixed penalty (lambda 0.05), SAAS and Soft-SAAS, seed 0 each;
5. evaluates every policy with greedy decoding on the five test sets (HotpotQA 1,000, NQ 500, 2WikiMultihopQA 1,000, MuSiQue 500, PopQA 500 questions), with the same commands as the original evaluation;
6. compares every answer with the recorded run and every EM, F1 and search count (SC) with the report's appendix table, and prints **PASS** or **FAIL**.

**How the comparison is judged**, per policy and test set:
* **exact**: every answer and every search count equals the recorded run. The two base model baselines without a recorded run here (no search, RAG) count as exact when EM, F1 and SC equal the report's table up to its rounding.
* **close**: at least 97% of answers equal, and EM and F1 within 1 point and SC within 0.05 of the recorded run.
* **differs**: anything else. Any "differs" makes the run FAIL.

## 1. Get the code

In [ ]:
!git clone -q --depth 1 https://github.com/Kodezilla0725/search_or_think.git /content/search_or_think 2>/dev/null || git -C /content/search_or_think pull -q
!git -C /content/search_or_think log -1 --format="code at commit %h, %cd"

## 2. Install the versions the evaluation used

In [ ]:
!pip install -q "trl[vllm,peft]==1.16.0" "vllm==0.31.0" pytest
!pip uninstall -y -q torchao  # as in the evaluation notebooks: Colab's torchao clashes with peft; nothing here uses it

## 3. Run the pipeline

Prints one line per step, the summary line of every evaluation, then the full results table and the verdict. The detailed log of each evaluation is in `/content/repro/logs/`.

In [ ]:
!cd /content/search_or_think && python -u reproduce.py --work /content/repro

## 4. Results

In [ ]:
import json

report = json.load(open("/content/repro/reproduce_report.json"))
print("VERDICT:", report["verdict"])
print("unit tests:", report.get("unit_tests", {}).get("summary"))
print("retrieval check:", {k: v for k, v in report.get("retrieval", {}).items() if k != "examples"})
print("minutes:", report["minutes"].get("total"))
not_exact = [r for r in report["rows"] if r["status"] != "exact"]
print(f"{len(report['rows']) - len(not_exact)} of {len(report['rows'])} policy and test set results are exact")
for r in not_exact:
    print(r)